In [ ]:
import csv
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import ast
import os
import torch
import torch.nn as nn
from tqdm import tqdm
from tqdm import tqdm
import torch.optim as optim
from sklearn.metrics import mean_absolute_error
import csv
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from efficientnet_pytorch import EfficientNet
import sys
sys.path.append("..")
import seaborn as sns
import plotly.graph_objs as go
import matplotlib.pyplot as plt
import torch.nn.functional as F
import pandas as pd
import mlflow
from scipy.signal import resample, detrend
from scipy.signal import butter, filtfilt

In [ ]:
df = pd.read_csv("../dataset/ppg_train_data (copy).csv")

In [ ]:
df_public = df.iloc[:1000]

In [ ]:
df_public.to_csv("ppg_public.csv")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

df = pd.read_csv('../results/res_ppg.csv',index_col=0)
# Example data
x = df['target']  # First measurement method
y = df['predict']   # Second measurement method

from pyCompare import blandAltman


conf_mat = confusion_matrix(x,y)

# Calculate the differences and means
diff = y - x
mean = (x + y) / 2

blandAltman(x, y,
            limitOfAgreement=1.96,
            confidenceInterval=80,
            confidenceIntervalMethod='approximate',
            detrend=None,
            percentage=False,
)

# Create the scatter plot with mean difference
plt.scatter(mean, diff)
plt.axhline(y=np.mean(diff), color='r', linestyle='--')  # Add mean difference line

# Customize the plot if needed (e.g., axis labels, title)
plt.xlabel('Mean of Methods')
plt.ylabel('Difference between Methods')
plt.title('Bland-Altman Plot')

# Show the plot
plt.show()


In [ ]:
# cmap
sns.heatmap(conf_mat,fmt='d',cmap='Greens')

#### FIX COMPOSED DATA

In [ ]:
val_targets = [] # List of target values for validation
val_data = []
with open('../dataset/val_data.csv', mode='r') as file:
    reader = csv.reader(file)
    for row in reader:
        val_data.append([float(x) for x in row])

with open('../dataset/val_target.csv', mode='r') as file:
    reader = csv.reader(file)
    for row in reader:
        val_targets.append([float(x) for x in row])


# train_targets = [] # List of target values for validation
# train_data = []
# with open('../dataset/train_data.csv', mode='r') as file:
#     reader = csv.reader(file)
#     for row in reader:
#         train_data.append([float(x) for x in row])

# with open('../dataset/train_target.csv', mode='r') as file:
#     reader = csv.reader(file)
#     for row in reader:
#         train_targets.append([float(x) for x in row])

In [ ]:
def preprocess_signal(sig, lowcut = 0.1 ,highcut = 3,fs = 75,order = 2):
        # Calculate nyquist frequency (half of sample rate)
        nyq = 0.5 * fs 

        # Calculate filter coefficients using Butterworth filter design
        b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')

        # Apply filter to PPG signal using filtfilt to avoid phase distortion
        filtered_ppg = filtfilt(b, a, -sig)
        
        # Enhance dicrotic notches by taking second derivative
        sig_ppg = np.gradient(np.gradient(filtered_ppg))
        # print(sig_ppg)
        return sig_ppg
    
# print(len(train_data))
# print(len(train_targets[0]))
print(len(val_data))
print(len(val_targets[0]))

public_data = val_data[:200]
public_data_label = val_targets[0][:200]


In [ ]:
remove_idx = np.where(np.array(public_data_label)<5)[0]

In [ ]:
print(len(public_data))
print(len(public_data_label))

In [ ]:
final_public_data = np.delete(public_data,remove_idx, axis=0)
final_public_data_label = np.delete(public_data_label,remove_idx, axis=0)

In [ ]:
with open(os.path.join('..','dataset','public_ecg_data.csv'), mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerows(final_public_data)
with open(os.path.join('..','dataset','public_ecg_target.csv'), mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(final_public_data_label)

In [ ]:
len(final_public_data_label)

In [ ]:
idx = np.random.randint(len(train_data))
fig = go.Figure()
fig.add_trace(
    go.Scatter(y=preprocess_signal(np.array(train_data[idx])), mode='lines')
)
fig.show()

In [ ]:
train_count_res = np.array([len(np.where(np.array(x) !=0)[0]) for x in train_data])
train_remove_index = (np.where(train_count_res<=2500)[0])

In [ ]:
train_remove_index

In [ ]:
new_train_data = [elem for i, elem in enumerate(train_data) if i not in train_remove_index]
print(len(new_train_data))   # Output: ['a', 'c', 'e']
new_train_target = [elem for i, elem in enumerate(train_targets[0]) if i not in train_remove_index]
print(len(new_train_target))   # Output: ['a', 'c', 'e']

In [ ]:
len(train_targets[0])

In [ ]:
# Create a histogram using sns.histplot()
sns.histplot(data=val_targets[0], kde=True)

# Display the plot
plt.show()

In [ ]:
# Create a histogram using sns.histplot()
sns.histplot(data=train_targets[0], kde=True)

# Display the plot
plt.show()

In [ ]:
# mask_sum = np.apply_along_axis(np.sum,1,train_data)
# mask = np.where(mask_sum==0)[0]
# train_data_new = np.delete(train_data,mask,axis=0)
# train_targets_new = np.delete(train_targets,mask)
# with open(os.path.join('..','dataset','ppg_train_data.csv'), mode='w', newline='') as file:
#     writer = csv.writer(file)
#     writer.writerows(train_data_new)
# with open(os.path.join('..','dataset','ppg_train_target.csv'), mode='w', newline='') as file:
#     writer = csv.writer(file)
#     writer.writerow(train_targets_new)

In [ ]:
# mask_sum = np.apply_along_axis(np.sum,1,val_data)
# mask = np.where(mask_sum==0)[0]
# val_data_new = np.delete(val_data,mask,axis=0)
# val_targets_new = np.delete(val_targets,mask)
# with open(os.path.join('..','dataset','ppg_val_data.csv'), mode='w', newline='') as file:
#     writer = csv.writer(file)
#     writer.writerows(val_data_new)
# with open(os.path.join('..','dataset','ppg_val_target.csv'), mode='w', newline='') as file:
#     writer = csv.writer(file)
#     writer.writerow(val_targets_new)

#### PLOT RESULTS and VISUALIZE METRICs

In [ ]:
df_res = pd.read_csv("results/res.csv",index_col=0)

In [ ]:
# Create a histogram using sns.histplot()
sns.histplot(data=df_res['target'], kde=True)

# Display the plot
plt.show()

In [ ]:
# Create a histogram using sns.histplot()
sns.histplot(data=df_res['predict'], kde=True)

# Display the plot
plt.show()

In [ ]:

# Create a histogram using sns.histplot()
sns.histplot(data=np.abs(df_res['target']-df_res['predict']), kde=True)

# Display the plot
mae=mean_absolute_error(df_res['target'],df_res['predict'])
plt.title(f"Plot the difference between prediction and ground truth, MAE = {mae:.2f}")
plt.show()

mean_absolute_error(df_res['target'],df_res['predict'])

In [ ]:
import sweetviz as sv
import dtale
# !pip install sweetviz
# !pip install dtale

In [ ]:
dtale.show(df_res)

In [ ]:
# report = sv.analyze(df_res)
# report.show_html('respiratory rate model report.html')

In [ ]:
target_values = df_res['target'].unique()
for value in target_values:
    print(f"================{value}==============")
    sns.histplot(data=df_res[df_res['target']==value]['target']-\
        df_res[df_res['target']==value]['predict'], kde=True)

    # Display the plot
    plt.show()

In [ ]:
# Create a histogram using sns.histplot()
sns.histplot(data=val_targets, kde=True)

# Display the plot
plt.show()


In [ ]:
# # Create a histogram using sns.histplot()
# sns.histplot(data=val_targets, kde=True)

# # Display the plot
# plt.show()
